# HTF策略模块演示

这个 notebook 演示如何使用 HTF 模块进行盘口失衡 + 成交确认短动量策略研究和回测。

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
sys.path.append('/home/jovyan/work/tactics_demo')
sys.path.append('/home/jovyan/work/tactics_demo/HTF')
sys.path.append('/home/jovyan/work/tactics_demo/tools')

## 导入HTF模块

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from feature import FeatureEngine, create_feature
from strategy import StrategyDemo
from delta.train import get_trade_dates

from parallel_backtest_simple import run_parallel_backtest
from multi_day_backtest import backtest_summary
from delay_stability_test import batch_delay_stability_test
from single_day_backtest import single_day_backtest

import base_tool

## 设置参数

In [ ]:
instrument_id = '518880'
trade_ymd = '20260319'

param_dict = {
    'name': 'htf_imbalance_trade_pressure',
    'instrument_id': instrument_id,
    'trade_ymd': trade_ymd,
    'imbalance_threshold': 0.35,
    'sustain_seconds': 5,
}

param_dict

## 获取交易日数据

In [ ]:
trade_dates = get_trade_dates()
print(f'总交易日数量: {len(trade_dates)}')
print(f'交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}')

## 检查单日特征

In [ ]:
snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
print(f'snapshot数量: {len(snap_list)}')

engine = FeatureEngine()

feature_rows = []
for snap in snap_list:
    feat = engine.update(snap)
    feat['time_mark'] = snap['time_mark']
    feat['price_last'] = snap['price_last']
    feature_rows.append(feat)

feature_df = pd.DataFrame(feature_rows)
feature_df.tail()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

feature_df[['imbalance_1_5']].plot(ax=axes[0], legend=False, title='imbalance_1_5')
feature_df[['trade_pressure_10s']].plot(ax=axes[1], legend=False, title='trade_pressure_10s')
feature_df[['mid_price', 'mid_ma_30s']].plot(ax=axes[2], title='mid_price vs mid_ma_30s')

plt.tight_layout()

## 创建策略实例

In [ ]:
strategy = StrategyDemo(None, param_dict)
print(f"策略已创建: {param_dict['name']}")

## 使用backtesting工具进行回测

In [ ]:
result_df = run_parallel_backtest(
    instrument_id=instrument_id,
    start_ymd='20260303',
    end_ymd='20260409',
    StrategyClass=StrategyDemo,
    model_path_or_obj=None,
    param_dict=param_dict,
    n_cores=4,
)

summary = backtest_summary(result_df)
summary

## 延迟回测结果

In [ ]:
batch_delay_stability_test(
    instrument_id,
    '20260303',
    '20260409',
    StrategyDemo,
    None,
    param_dict,
    delay_list=[0, 1, 2, 3, 5],
    n_cores=4,
)

## 单日回测观察

In [ ]:
hh = single_day_backtest(
    instrument_id,
    trade_ymd,
    StrategyDemo,
    None,
    param_dict,
    official=False,
)
hh